# Agilent BioTek PowerWaveX

```{device-card} biotek-powerwave-x
```

| Property | Value |
|---|---|
| Connection | RS-232 serial, 9600 8N2 |
| PLR extra | `serial` |
| Modes | Absorbance: endpoint, kinetic and spectral scan |
| Wavelengths | 200-999 nm, 1 nm increments, up to six per read |
| Plates | 96-well (8x12) and 384-well (16x24) |
| Incubator | Optional, 22-50 degrees C, model dependent |

:::{warning}
Absorbance **data** and **spectral scanning** have not been confirmed against an instrument. Everything else
here has: identification, the wavelength table, temperature, carrier movement, the
assay definition, plate geometry, strip range and the self-test were all exercised
on a PowerWaveX w/ Incubator running firmware R06.20. That instrument's optics fail
their self-test, so no usable readings could be taken from it; the parsing is
verified against serial traffic captured from the same machine in 2021, when its
optics still worked. Please report your results at
[discuss.pylabrobot.org](https://discuss.pylabrobot.org).
:::

The PowerWaveX is an absorbance-only reader from the early 2000s. It predates the
USB BioTek readers and speaks a different command language, so it is a separate
class from {class}`~pylabrobot.agilent.biotek.Cytation5` and
{class}`~pylabrobot.agilent.biotek.SynergyH1` rather than a subclass.

Install the serial extra:

```
pip install pylabrobot[serial]
```

## Communication

The instrument speaks a byte-oriented protocol over RS-232 at 9600 baud, 8 data
bits, no parity and **two** stop bits. Each command is a single ASCII character
answered with `ACK` or `NAK`, usually followed by a five-byte status string.

Wavelengths are selected by a monochromator, so the stored wavelength table is a
*calibration* set rather than a list of installed filters. Reads at other
wavelengths work and simply warn that gains are uncalibrated there.

## Physical setup

Connect a USB-to-RS-232 adapter to the instrument's serial port and power it on.
On Linux the port is typically `/dev/ttyUSB0`; your user needs access to it, which
usually means membership of the `uucp` or `dialout` group. On Windows it is a `COM`
port.

## Create the driver

Pass the serial port, or a USB vendor and product id to have it discovered.

In [ ]:
from pylabrobot.agilent.biotek import PowerWaveX

reader = PowerWaveX(name="reader", port="/dev/ttyUSB0")

## Connect

`setup()` opens the port and identifies the instrument, reading its configuration
word and calibrated wavelength table.

In [ ]:
await reader.setup()

## Identify the instrument

Both values are cached at setup, so these do not talk to the instrument again.

In [ ]:
print(reader.model)
print(reader.wavelength_table)
print(reader.supports_heating)

## Move the plate carrier

`open()` presents the carrier so a plate can be loaded, `close()` stores it inside.
Both are idempotent: calling `open()` twice sends nothing the second time.

In [ ]:
await reader.open()
# load a plate
await reader.close()

## Incubator

The setpoint is whole degrees, 22 to 50, or 0 to switch the heaters off. The
instrument offers no way to read its setpoint back, and keeps it in non-volatile
memory across power cycles, so a value left by an earlier session is invisible
until the chamber temperature reveals it.

In [ ]:
await reader.set_temperature(37)
print(await reader.request_current_temperature())

## Read absorbance

Pass the plate, the wells to read, and a wavelength in nm. The result is a list of
{class}`~pylabrobot.agilent.biotek.plate_reader_base.AbsorbanceResult`, whose
`data` is indexed `[row][column]` in optical density units.

In [ ]:
from pylabrobot.resources import cellvis_96_wellplate_350uL_Fb

plate = cellvis_96_wellplate_350uL_Fb(name="plate")

results = await reader.read_absorbance(plate, plate.get_all_items(), 630)
print(results[0].data[0][0])  # well A1

## Read several wavelengths in one pass

The assay definition holds up to six wavelengths and a single pass of the plate
returns one data block for each, so this is far quicker than reading one at a
time.

In [ ]:
results = await reader.read_absorbances(plate, plate.get_all_items(), [450, 630, 990])

for result in results:
    print(result.wavelength, result.data[0][0])

## Read part of a plate

The instrument reads whole columns, called strips, and cannot select rows. A
selection that does not fill every row of its columns is widened to the enclosing
strips, and those extra wells are reported rather than discarded. Columns outside
the range were never measured and come back as `None`.

In [ ]:
wells = [w for w in plate.get_all_items() if w.get_column() < 3]

results = await reader.read_absorbance(plate, wells, 630)
print(results[0].data[0][0])   # measured
print(results[0].data[0][11])  # None, never read

## Kinetic read

The instrument runs the whole series itself, emitting one data block per
wavelength per cycle, so this call blocks for roughly `interval * count` seconds.
Each result carries the timestamp of its cycle.

`request_minimum_kinetic_interval()` reports the shortest interval the loaded assay
allows; a shorter request is accepted but warns, and cycles will run late.

In [ ]:
results = await reader.read_absorbance_kinetic(
    plate,
    plate.get_all_items(),
    wavelengths=[600, 630],
    interval=60,
    count=30,
)

for result in results[:4]:
    print(result.timestamp, result.wavelength, result.data[0][0])

## Spectral scan

A scan sweeps the monochromator across a range for **one well or one column**,
rather than reading the whole plate at a few fixed wavelengths. Use it to find
an absorbance maximum, or to check a spectrum.

At least two wavelengths must be scanned, so the step cannot exceed the span.

In [ ]:
result = await reader.scan_plate(
    plate,
    start_wavelength=400,
    stop_wavelength=700,
    step=10,
    well=plate.get_item(0),  # A1
)

for wavelength, absorbance in result.spectrum():
    print(wavelength, absorbance)

### Scanning a whole column

Passing `column` instead of `well` scans every row of that column at once. The
instrument uses two passes internally, because even-row detectors sit one column
to the left; the driver handles that transparently.

`data` is then indexed `[wavelength][row]`.

In [ ]:
result = await reader.scan_plate(
    plate, start_wavelength=400, stop_wavelength=700, step=25, column=1
)

print(result.wavelengths)
print(result.data[0])        # every row at the first wavelength
print(result.spectrum(3))    # the spectrum of row D

### Calibration

`calibrate=True` calibrates every wavelength before scanning, which the manual
recommends when absolute accuracy matters. Without it the instrument still
calibrates any wavelength not scanned since power-up, and any that previously
errored. `result.calibrated` reports what it actually did.

In [ ]:
result = await reader.scan_plate(
    plate,
    start_wavelength=400,
    stop_wavelength=700,
    step=10,
    well=plate.get_item(0),
    calibrate=True,
)
print(result.calibrated)

## Abort a read

`halt()` stops a read or scan in progress, homes every axis and presents the
carrier outside.

In [ ]:
await reader.halt()

## Self-test

Run this when a read fails with status `080`, and after changing the wavelength
table, since it recalibrates gains. It takes a couple of minutes and moves every
axis.

The self-test's status string reports only whether the *command* ran: a failing
instrument still answers `000`. The driver therefore takes the verdict from the
report body and raises `PowerWaveSelfTestError` with the decoded error code.

In [ ]:
from pylabrobot.agilent.biotek.powerwave import PowerWaveSelfTestError

try:
    print(await reader.self_test())
except PowerWaveSelfTestError as e:
    print(e.code, e.description)

## Disconnect

In [ ]:
await reader.stop()